<h1>Merging the different data sets</h1>
<p>In the following file it is merge the socio-economic data set orginzed by city, state and county with the final 2020 presidential election results organized by county, state.</p>

In [35]:
import pandas as pd 
import numpy as np

In [36]:
socio_data=pd.read_csv('Socioeconomic Data (2014 - 2018).csv')
socio_data=socio_data[(socio_data['State']!='PR') & (socio_data['Population']!=0) ]

In [37]:
#read the data that contains relationship beetween zip and county
key=pd.read_csv('uszips.csv')
#include a county column in the socioeconomic dataset 
def include_counties(zipcode,city):
    if zipcode in key.zip.unique():
        return key[key['zip']==zipcode]['county_name'].iloc[0]
    elif city.title() in key.city.unique():
        return key[key['city']==city.title()]['county_name'].iloc[0]
    else:
        return 'None'
    
socio_data.insert(1,'county_name',socio_data.apply(lambda x: include_counties(x['ZipCode'],x['City']),axis=1))

In [38]:
#check that all data has an associated county 
not_county=socio_data[socio_data['county_name']=='None']
len(not_county)
# socio_data[socio_data.county_name=='Abbeville']

0

In [39]:
#since we have more cities per county in the socio-economic data, but we dont have the votes per city, we organized all the information 
# by county, taking the mean of all the socioeconomic data from all the cities in each county 
notuseful_features=['Population','Predominant race','Total population of state','ZipCode','City']
socio_data=socio_data.drop(notuseful_features,axis=1).groupby(['county_name','State']).mean()
socio_data.to_excel('socio_data.xlsx')

<h2>Votes Data</h2>
<p>Prepare votes data to merge it with the original data set, changing the county written format and including the state_id  </p>

In [40]:
votes_county=pd.read_csv('president_county_candidate.csv')
#get list of County in approximately the same format than our key data
def simplifier(string):
    if 'County' == string[-6:]:
        return string[:-7]
    else:
        return string
votes_county['county']=votes_county.apply(lambda x: simplifier(x['county']),axis=1)
list_counties=list(votes_county.county.unique())

#include the state id 
state_key=key[['state_id','state_name']]
def get_state_id(state):
    if state in list(state_key.state_name.unique()):
        return state_key[state_key['state_name']==state]['state_id'].iloc[0]
    else:
        return 'None'
votes_county['state']=[get_state_id(state) for state in votes_county.state]
votes_county=votes_county.groupby(['county','state','party']).sum()

In [41]:
#Since both lists have different formats for naming counties, we define this function to help match counties based on words. 
def find_county(word):
    for county in list_counties:
        if county.find(word)!=-1:
            return county

def get_votes(county,state,party):
    if (county,state,party) in votes_county.index:
        return votes_county.loc[(county,state,party)]['votes']
    else:
        county=find_county(county)
        if (county,state,party) in votes_county.index:
            return votes_county.loc[(county,state,party)]['votes']
        else:
            return 'None'
        
final_data=socio_data.reset_index()
parties=['DEM','REP']
for party in parties:
    final_data.insert(0,party,final_data.apply(lambda x: get_votes(x['county_name'],x['State'],party),axis=1))

In [42]:
#missing infomation 
#there are few county names where we do not have vote information (80 out of 3128) 
final_data=final_data[final_data.REP!='None']

In [43]:
final_data.insert(2, 'Registerd',final_data['REP']+final_data['DEM'])


In [44]:
final_data['REP']=final_data['REP']/final_data['Active voters']
final_data['DEM']=final_data['DEM']/final_data['Active voters']
final_data

,REP,DEM,Registerd,county_name,State,% age 25-34,% age 35-44,% age 45-54,% age 55-64,% age 65 or older,...,Average HH size,Median gross rent,Median value of an owner-occupied home,Median HH income,Median owner cost burden,Median renter cost burden,% Asian,% Black,% Hispanic,% White
0,0.646619,0.353381,14449,Abbeville,SC,10.288667,10.517500,11.790333,16.543667,24.078000,...,2.502167,728.833333,95739.999994,39557.233334,15.391667,33.148333,1.037333,9.143500,3.617500,86.376667
1,0.805878,0.194122,28039,Acadia,LA,14.357708,12.211417,12.420708,12.759250,13.154000,...,2.827750,667.325000,118608.333330,49745.220833,12.301250,26.236250,0.387383,17.377533,2.416433,79.292550
2,0.547582,0.452418,16750,Accomack,VA,11.056632,11.274000,13.029625,16.301854,21.739076,...,2.373333,797.656250,152720.486112,40916.204861,16.405556,28.705208,3.330545,32.345016,6.887462,58.932408
3,0.52022,0.47978,251238,Ada,ID,13.846994,13.671397,13.513508,12.722110,14.538925,...,2.534280,1041.757567,258447.737760,65698.628093,17.986236,27.683560,1.134933,0.454400,3.525526,93.455696
4,0.709042,0.290958,4114,Adair,IA,10.295694,10.963889,12.862500,16.609722,20.438611,...,2.269861,657.500000,114511.111117,57052.472222,15.975000,23.056944,1.824444,3.026111,9.036944,89.092500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3123,0.532717,0.467283,65609,Yuma,AZ,11.690674,11.913663,10.508008,11.467101,17.912618,...,2.724683,744.475000,110821.416670,43137.655834,16.539417,29.529167,0.143000,1.653917,29.900750,90.843000
3124,0.841042,0.158958,4800,Yuma,CO,11.804048,10.887619,12.432619,12.498810,17.883333,...,3.126667,757.547619,171045.238100,49239.690476,14.752381,26.345238,2.726905,6.264708,18.185628,78.008528
3125,0.527518,0.472482,3852,Zapata,TX,11.562222,10.762222,10.167778,8.155556,12.602222,...,2.768889,494.777778,66300.000000,29566.888890,13.233333,32.866667,0.379653,1.075947,62.138507,87.103829
3126,0.342214,0.657786,4354,Zavala,TX,11.677222,12.371111,12.193889,10.218333,12.681667,...,3.380000,601.444444,49644.444443,25880.777777,13.372222,32.572222,0.068333,0.246667,40.903333,91.505000


In [47]:
final_data.to_csv('all_merged_data.csv')

In [48]:
final_data[final_data.State!='GA'].to_csv('training_data.csv')
final_data[final_data.State=='GA'].to_csv('Georgia_data.csv')
